Medical question classifier

## Step 1 Loading the Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings

warnings.filterwarnings('ignore')

# df = pd.read_csv("/content/medquad.csv")
df = pd.read_csv("../data/raw/medquad.csv")

# First look of the dataset
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print()
print(df.dtypes)
print()
print(df.head(3))

### Step 2 :: Checking missing values

In [ ]:
# Count nulls per column
print("Missing values:")
print(df.isnull().sum())

# Show percentage missing
print()
print("Missing %:")
missing_pct = df.isnull().sum() / len(df) * 100
print(missing_pct.round(2))

# Show which rows have missing answer
print()
print("Rows with missing answer:")
print(df[df['answer'].isnull()][['question', 'source']])

### Step -3 Removing Duplicates

In [ ]:
# Check before removing
print("Before:", df.shape)
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate questions:", df['question'].duplicated().sum())

# Step 1 — drop fully identical rows
df = df.drop_duplicates()

# Step 2 — drop repeated questions, keep first
df = df.drop_duplicates(subset='question', keep='first')

# Reset index after dropping
df = df.reset_index(drop=True)

print("After:", df.shape)

### Step 4  Drop missing rows

In [ ]:
# Droping rows where answer is null
df = df.dropna(subset=['answer'])

# Droping rows where focus_area is null
df = df.dropna(subset=['focus_area'])

# Reset index
df = df.reset_index(drop=True)

# Confirming no nulls remain
print("Shape after dropping nulls:", df.shape)
print("Any nulls remaining?", df.isnull().any().any())
print(df.isnull().sum())

### Step -5 Cleaning the question text

In [ ]:
# Define the cleaning function
def clean_text(text):

    # lowercase everything
    text = text.lower()

    #  removing (are) (s) (es) patterns
    text = re.sub(r'\(are\)|\(s\)|\(es\)', '', text)

    # removing all special characters
    text = re.sub(r'[^a-z0-9\s]', '', text)

    # collaping multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text


# Applying to question column
df['question_clean'] = df['question'].apply(clean_text)


# Showing before vs after for first 3 rows
for i in range(3):
    print(f"BEFORE: {df['question'][i]}")
    print(f"AFTER : {df['question_clean'][i]}")
    print()

### Step 6 Creating the qtype label

In [ ]:
# Define label extraction function
def extract_qtype(q):

    # Convert to lowercase
    q = q.lower()

    if re.search(r'symptom', q):
        return 'symptoms'

    if re.search(r'treatment|therapy|treat', q):
        return 'treatment'

    if re.search(r'cause', q):
        return 'causes'

    if re.search(r'diagnos|how to test', q):
        return 'diagnosis'

    if re.search(r'prevent', q):
        return 'prevention'

    if re.search(r'genetic|inherit', q):
        return 'genetic'

    if re.search(r'risk factor', q):
        return 'risk_factors'

    if re.search(r'prognosis|outlook', q):
        return 'prognosis'

    if re.search(r'how many|statistic', q):
        return 'epidemiology'

    # Default labeling
    return 'definition'


# Applying to question_clean column
df['qtype'] = df['question_clean'].apply(extract_qtype)

# Check resulting
print(df['qtype'].value_counts())


## Step 7 EDA on visualise distributions

In [ ]:
# Add word count column
df['q_words'] = df['question_clean'].str.split().str.len()

# Print word count stats
print("Word count stats:")
print(df['q_words'].describe().round(1))
print()

# Create figure with 2 plots side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1 — label distribution (horizontal bar)
counts = df['qtype'].value_counts()
counts.plot(kind='barh', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Label distribution (qtype)')
axes[0].set_xlabel('Count')

# Plot 2 — question word count histogram
df['q_words'].hist(bins=25, ax=axes[1], color='teal', edgecolor='white')
axes[1].set_title('Question word count')
axes[1].set_xlabel('Words per question')
axes[1].set_ylabel('Count')

# Save and show
plt.tight_layout()
plt.savefig('../reports/figures/eda_plots.png', dpi=120)
plt.show()

## Step 8 Adding department and saving dataset

In [ ]:
import re

#  Keyword rules: most specific first, General last
RULES = [
  ('Oncology', [r'cancer', r'carcinoma', r'tumor', r'leukemia',
   r'lymphoma', r'sarcoma', r'melanoma', r'myeloma',
   r'blastoma', r'neoplasm', r'malignant', r'oncol']),

  ('Cardiology', [r'heart', r'cardiac', r'cardio', r'arrhythmia',
   r'coronary', r'angina', r'aortic', r'hypertension',
   r'blood pressure', r'cholesterol', r'atherosclerosis']),

  ('Neurology', [r'alzheimer', r'parkinson', r'stroke', r'epilepsy',
   r'seizure', r'multiple sclerosis', r'dementia', r'migraine',
   r'brain', r'spinal cord', r'neuropathy', r'tremor']),

  ('Orthopedics & Rheumatology', [r'arthritis', r'osteoporosis',
   r'bone', r'joint', r'fracture', r'scoliosis', r'lupus',
   r'fibromyalgia', r'gout', r'spondyl']),

  ('Pulmonology', [r'lung', r'pulmonar', r'respiratory', r'copd',
   r'asthma', r'pneumonia', r'bronchi', r'tuberculosis']),

  ('Gastroenterology & Hepatology', [r'gastro', r'digestive',
   r'liver', r'hepat', r'colon', r'bowel', r'stomach',
   r'pancrea', r'crohn', r'gallbladder']),

  ('Nephrology & Urology', [r'kidney', r'renal', r'nephro',
   r'urolog', r'urinary', r'bladder', r'prostate', r'dialysis']),

  ('Endocrinology & Metabolism', [r'diabetes', r'thyroid',
   r'adrenal', r'hormone', r'insulin', r'obesity', r'metabolic',
   r'pituitary', r'glucose', r'lysosomal storage']),

  ('Hematology', [r'anemia', r'hemoglobin', r'hemophilia',
   r'platelet', r'thrombosis', r'bleeding disorder',
   r'bone marrow', r'sickle cell', r'thalassemia']),

  ('Ophthalmology', [r'eye', r'vision', r'ocular', r'retina',
   r'glaucoma', r'cataract', r'macular', r'cornea', r'optic']),

  ('Psychiatry & Mental Health', [r'anxiety', r'depression',
   r'bipolar', r'schizophrenia', r'mental', r'addiction',
   r'ptsd', r'autism', r'adhd', r'suicide']),

  ('Dermatology', [r'skin', r'derma', r'eczema', r'psoriasis',
   r'acne', r'alopecia', r'hair loss', r'rash', r'vitiligo']),

  ('Infectious Disease', [r'infection', r'hiv', r'aids',
   r'bacteria', r'viral', r'malaria', r'dengue', r'sepsis',
   r'lyme', r'immunization', r'vaccine']),

  ('Gynecology & Obstetrics', [r'pregnancy', r'obstetric',
   r'ovarian', r'uterine', r'menopause', r'cervical',
   r'fertility', r'endometriosis', r'breastfeeding']),

  ('Pediatrics', [r'child', r'pediatric', r'infant', r'newborn',
   r'adolescent', r'teen', r'childhood', r'juvenile']),

  ('Geriatrics & Aging', [r'senior', r'elderly', r'older adult',
   r'aging', r'geriatric', r'nursing home', r'medicare']),

  ('Immunology & Allergy', [r'allerg', r'autoimmune', r'immune',
   r'immunodeficiency', r'anaphylaxis', r'mast cell']),

  ('ENT', [r'ear', r'nose', r'throat', r'sinus', r'hearing',
   r'tonsil', r'laryn', r'dental', r'tooth', r'tinnitus']),

  ('Genetics & Rare Diseases', [r'syndrome', r'genetic',
   r'hereditary', r'congenital', r'chromosomal', r'dysplasia']),

  ('General Medicine & Public Health', [r'.*']),
]

def classify_department(focus_area):
  if pd.isna(focus_area):
    return 'General Medicine & Public Health'
  text = str(focus_area).lower()
  for dept, patterns in RULES:
    for pat in patterns:
      if re.search(pat, text, re.IGNORECASE):
        return dept
  return 'General Medicine & Public Health'

#  Apply mapping
df['department'] = df['focus_area'].apply(classify_department)

# Keep only the 4 columns the model needs
df_model = df[['question_clean', 'qtype', 'focus_area', 'department']].copy()

#  Save
# df_model.to_csv('../data/processed/medquad_model_ready.csv', index=False)
df_model.to_csv('../data/processed/medquad_cleaned_ready(1).csv', index=False)

#  Confirm
print("Saved: medquad_cleaned_ready(1).csv")
print(f"Final shape: {df_model.shape}")
print(f"Unique departments: {df_model['department'].nunique()}")
print()
print(df_model['department'].value_counts())